# PolyChain: Polymer Property Prediction
## AISEHack 2.0 - Kaggle Pipeline

This notebook runs the full PolyChain pipeline: data loading, feature extraction,
cross-validation training, ensemble blending, and submission generation.

In [ ]:
# Cell 1: Install dependencies
!pip install -q rdkit-pypi pandas numpy scikit-learn pyyaml\
  torch xgboost catboost lightgbm fastparquet

In [ ]:
# Cell 2: Environment setup
import sys, os, shutil
from pathlib import Path

# Detect Kaggle environment
KAGGLE = Path("/kaggle").exists()
if KAGGLE:
    DATA_SOURCE = Path("/kaggle/input/aisehack-2-0")
    WORK_DIR = Path("/kaggle/working/polymer_competition")
    sys.path.insert(0, str(WORK_DIR))
else:
    DATA_SOURCE = Path("data")
    WORK_DIR = Path.cwd()

os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")
print(f"Data source: {DATA_SOURCE}")

In [ ]:
# Cell 3: Copy competition data
if KAGGLE:
    shutil.copy(DATA_SOURCE / "train.csv", "data/train.csv")
    shutil.copy(DATA_SOURCE / "test.csv", "data/test.csv")
    print("Data copied from Kaggle input")
else:
    print("Using local data")

# Verify
import pandas as pd
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(f"Train: {len(train)} rows, columns: {list(train.columns)}")
print(f"Test: {len(test)} rows, columns: {list(test.columns)}")

In [ ]:
# Cell 4: Split data by target type
from data.split_by_target import split_by_target
split_by_target("data/train.csv", "data/test.csv", "data/")
print("Per-target splits created: data/tg/*.csv, data/egc/*.csv")

In [ ]:
# Cell 5: Build features (fingerprints + descriptors + polymer features)
from features.build_features import build_features
build_features("config.yaml")
print("Feature cache built")

In [ ]:
# Cell 6: Generate CV splits per target
from data.generate_splits import generate_splits
for target in ["tg", "egc"]:
    generate_splits(
        f"data/{target}/train.csv",
        f"data/splits_{target}.pkl",
        n_folds=5, smiles_col="smiles", target_col="target",
    )
    print(f"Splits done for {target}")

In [ ]:
# Cell 7: Train selected models per target
import subprocess, sys, yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
n_folds = cfg["cv"]["n_folds"]

# Models to train (adjust as needed for time limits)
MODELS = ["ridge", "xgb", "lgb"]
TARGETS = ["tg", "egc"]

for target in TARGETS:
    for model_type in MODELS:
        for fold in range(n_folds):
            cmd = [
                sys.executable, "-m", "training.train",
                "--model_type", model_type,
                "--fold", str(fold),
                "--config", "config.yaml",
                "--target", target,
            ]
            result = subprocess.run(cmd, cwd=str(WORK_DIR))
            if result.returncode != 0:
                print(f"WARNING: {model_type} {target} fold {fold} failed")
            else:
                print(f"OK: {model_type} {target} fold {fold}")

In [ ]:
# Cell 8: Build ensembles per target and merge submissions
for target in TARGETS:
    result = subprocess.run([
        sys.executable, "-m", "ensemble.build_ensemble",
        "--config", "config.yaml", "--target", target,
    ], cwd=str(WORK_DIR))
    print(f"Ensemble for {target}: {'OK' if result.returncode == 0 else 'FAILED'}")

# Merge per-target submissions into final submission
result = subprocess.run([
    sys.executable, "-m", "data.merge_submissions",
    "--config", "config.yaml",
], cwd=str(WORK_DIR))
print(f"Submission merge: {'OK' if result.returncode == 0 else 'FAILED'}")

In [ ]:
# Cell 9: Copy submission to Kaggle output
sub_path = Path("outputs/submissions/submission.csv")
if sub_path.exists():
    shutil.copy(str(sub_path), "/kaggle/working/submission.csv")
    print(f"Submission copied to /kaggle/working/submission.csv")
else:
    print(f"Submission not found at {sub_path}")

# Display submission preview
sub = pd.read_csv("/kaggle/working/submission.csv" if KAGGLE else str(sub_path))
print(f"Submission rows: {len(sub)}")
print(sub.head())

In [ ]:
# Cell 10: Show experiment manifest (optional)
import json
manifest_path = Path("experiments/manifest.json")
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"Completed runs: {len(manifest)}")
    for r in manifest[-3:]:
        print(f"  {r['target']} {r['model']} fold {r['fold']}: r2={r['score']:.4f}")